# YDL 2026 Day 4 — Titanic with **Linear Regression only**

**Task:** predict Titanic survival on the Kaggle test set using *only* `sklearn.linear_model.LinearRegression`.

Survival is binary, but the constraint forces us to use a regressor. We therefore:
1. Engineer strong features (Title, FamilySize, HasCabin, Deck, TicketGroupSize, FarePerPerson, …).
2. Fill missing values smartly (grouped medians, not one global mean).
3. One-hot encode categoricals and scale features.
4. Fit `LinearRegression`, then **tune a decision threshold** with repeated stratified cross-validation.
5. Retrain on the full training set and write the Kaggle submission.

Competition: https://www.kaggle.com/competitions/titanic

## 1. Imports

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression          # the ONLY ML model allowed
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

## 2. Load data

Notebook lives in `notebooks/`, so data is one level up in `../data/`. We keep the test `PassengerId` for the submission.

In [2]:
DATA = '../data' if os.path.isdir('../data') else 'data'
SUB = '../submissions' if os.path.isdir('../submissions') else 'submissions'
os.makedirs(SUB, exist_ok=True)

train = pd.read_csv(os.path.join(DATA, 'train.csv'))
test = pd.read_csv(os.path.join(DATA, 'test.csv'))
passenger_id = test['PassengerId']

print('train', train.shape, '| test', test.shape)
train.head()

train (891, 12) | test (418, 11)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 3. EDA

In [3]:
print('Missing values (train):')
print(train.isna().sum()[train.isna().sum() > 0])
print('\nMissing values (test):')
print(test.isna().sum()[test.isna().sum() > 0])
print('\nOverall survival rate:', round(train['Survived'].mean(), 4))

Missing values (train):
Age         177
Cabin       687
Embarked      2
dtype: int64

Missing values (test):
Age       86
Fare       1
Cabin    327
dtype: int64

Overall survival rate: 0.3838


In [4]:
print('Survival by Sex:')
print(train.groupby('Sex')['Survived'].mean(), '\n')
print('Survival by Pclass:')
print(train.groupby('Pclass')['Survived'].mean(), '\n')
print('Survival by Embarked:')
print(train.groupby('Embarked')['Survived'].mean())

Survival by Sex:
Sex
female    0.742038
male      0.188908
Name: Survived, dtype: float64 

Survival by Pclass:
Pclass
1    0.629630
2    0.472826
3    0.242363
Name: Survived, dtype: float64 

Survival by Embarked:
Embarked
C    0.553571
Q    0.389610
S    0.336957
Name: Survived, dtype: float64


## 4–7. Feature engineering, missing values, encoding

One shared function builds features for train and test consistently. Train and test are concatenated only to compute **fills** and **ticket-group sizes** — this uses no target values, so there is no leakage.

- **Title** extracted from `Name`; rare titles folded into `Rare`; `Mlle/Ms→Miss`, `Mme→Mrs`.
- **Family:** `FamilySize`, `IsAlone`, `SmallFamily`, `LargeFamily`.
- **Cabin:** `HasCabin` flag + `Deck` letter.
- **Ticket:** `TicketGroupSize` (people sharing a ticket).
- **Fare:** median fill by `Pclass`, plus `FarePerPerson` and `log1p(Fare)`.
- **Age:** median by `Title`+`Pclass`, with fallbacks; `IsChild` flag.
- Categoricals one-hot encoded; train/test columns aligned.

In [5]:
TITLE_MAP = {
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Lady': 'Rare', 'Countess': 'Rare', 'Capt': 'Rare', 'Col': 'Rare',
    'Don': 'Rare', 'Dr': 'Rare', 'Major': 'Rare', 'Rev': 'Rare',
    'Sir': 'Rare', 'Jonkheer': 'Rare', 'Dona': 'Rare',
}

def extract_title(name):
    title = name.split(',')[1].split('.')[0].strip()
    return TITLE_MAP.get(title, title)


def build_features(train, test):
    """Return (X_train, X_test, y) as numeric, column-aligned matrices."""
    y = train['Survived'].astype(float).values

    train = train.drop(columns=['Survived']).copy()
    train['__src__'] = 'train'
    test = test.copy()
    test['__src__'] = 'test'
    df = pd.concat([train, test], ignore_index=True)

    # Title
    df['Title'] = df['Name'].apply(extract_title)
    df.loc[~df['Title'].isin(['Mr', 'Mrs', 'Miss', 'Master']), 'Title'] = 'Rare'

    # Family
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['SmallFamily'] = df['FamilySize'].between(2, 4).astype(int)
    df['LargeFamily'] = (df['FamilySize'] >= 5).astype(int)

    # Cabin / Deck
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['Deck'] = df['Cabin'].astype(str).str[0]
    df.loc[df['Cabin'].isna(), 'Deck'] = 'Unknown'
    df.loc[df['Deck'] == 'T', 'Deck'] = 'Unknown'  # single-row deck

    # Ticket group size (across train + test)
    df['TicketGroupSize'] = df.groupby('Ticket')['Ticket'].transform('count')

    # Fare
    df['Fare'] = df.groupby('Pclass')['Fare'].transform(lambda s: s.fillna(s.median()))
    df['FarePerPerson'] = df['Fare'] / df['TicketGroupSize']
    df['FareLog'] = np.log1p(df['Fare'])

    # Embarked
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    # Age: median by Title+Pclass, with fallbacks
    df['Age'] = df.groupby(['Title', 'Pclass'])['Age'].transform(lambda s: s.fillna(s.median()))
    df['Age'] = df.groupby(['Title'])['Age'].transform(lambda s: s.fillna(s.median()))
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['IsChild'] = (df['Age'] < 16).astype(int)

    # Sex
    df['Sex'] = (df['Sex'] == 'male').astype(int)

    numeric = [
        'Sex', 'Age', 'FareLog', 'FarePerPerson',
        'FamilySize', 'IsAlone', 'SmallFamily', 'LargeFamily',
        'HasCabin', 'TicketGroupSize', 'IsChild',
    ]
    categorical = ['Embarked', 'Title', 'Deck', 'Pclass']

    encoded = pd.get_dummies(df[numeric + categorical], columns=categorical,
                             drop_first=False, dtype=float)

    X = encoded[df['__src__'].values == 'train'].reset_index(drop=True)
    X_test = encoded[df['__src__'].values == 'test'].reset_index(drop=True)
    X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)
    return X, X_test, y


X, X_test, y = build_features(train, test)
print('Feature matrix: train', X.shape, '| test', X_test.shape)
print('Features:', list(X.columns))
X.head()

Feature matrix: train (891, 30) | test (418, 30)
Features: ['Sex', 'Age', 'FareLog', 'FarePerPerson', 'FamilySize', 'IsAlone', 'SmallFamily', 'LargeFamily', 'HasCabin', 'TicketGroupSize', 'IsChild', 'Embarked_C', 'Embarked_Q', 'Embarked_S', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_Unknown', 'Pclass_1', 'Pclass_2', 'Pclass_3']


,Sex,Age,FareLog,FarePerPerson,FamilySize,IsAlone,SmallFamily,LargeFamily,HasCabin,TicketGroupSize,IsChild,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Deck_A,Deck_B,Deck_C,Deck_D,Deck_E,Deck_F,Deck_G,Deck_Unknown,Pclass_1,Pclass_2,Pclass_3
0,1,22.0,2.110213,7.25000,2,0,1,0,0,1,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
1,0,38.0,4.280593,35.64165,2,0,1,0,1,2,0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0,26.0,2.188856,7.92500,1,1,0,0,0,1,0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
3,0,35.0,3.990834,26.55000,2,0,1,0,1,2,0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,1,35.0,2.202765,8.05000,1,1,0,0,0,1,0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0


## 8. Train / validation split (quick sanity check)

In [6]:
Xv = X.values.astype(float)
Xtv = X_test.values.astype(float)

X_tr, X_val, y_tr, y_val = train_test_split(
    Xv, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

scaler = StandardScaler().fit(X_tr)
model = LinearRegression().fit(scaler.transform(X_tr), y_tr)
val_raw = model.predict(scaler.transform(X_val))
print('Hold-out accuracy @0.5:', round(accuracy_score(y_val, (val_raw >= 0.5).astype(int)), 4))

Hold-out accuracy @0.5: 0.8324


## 9–10. LinearRegression + threshold tuning via repeated stratified CV

A single hold-out split is noisy for picking a threshold, so we average accuracy across **5-fold × 10 repeats = 50 folds** for each candidate threshold and pick the best.

In [7]:
def tune_threshold(X, y):
    thresholds = np.arange(0.20, 0.801, 0.01)
    cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=RANDOM_STATE)
    acc_sum = np.zeros(len(thresholds))
    n = 0
    for tr_idx, val_idx in cv.split(X, y):
        sc = StandardScaler().fit(X[tr_idx])
        m = LinearRegression().fit(sc.transform(X[tr_idx]), y[tr_idx])
        raw = m.predict(sc.transform(X[val_idx]))
        for i, t in enumerate(thresholds):
            acc_sum[i] += accuracy_score(y[val_idx], (raw >= t).astype(int))
        n += 1
    mean_acc = acc_sum / n
    best_i = int(np.argmax(mean_acc))
    return thresholds[best_i], mean_acc[best_i], thresholds, mean_acc


best_t, best_acc, thresholds, mean_acc = tune_threshold(Xv, y)
print(f'Best CV threshold = {best_t:.2f}  |  CV accuracy = {best_acc:.4f}')
print('CV accuracy @0.50  =', round(mean_acc[np.argmin(np.abs(thresholds - 0.50))], 4))

Best CV threshold = 0.58  |  CV accuracy = 0.8297
CV accuracy @0.50  = 0.8277


## 11. Retrain on the full training set

In [8]:
scaler = StandardScaler().fit(Xv)
model = LinearRegression().fit(scaler.transform(Xv), y)
test_raw = model.predict(scaler.transform(Xtv))
test_pred = (test_raw >= best_t).astype(int)
print('Predicted survivors:', int(test_pred.sum()), '/', len(test_pred))

Predicted survivors: 156 / 418


## 12. Create the Kaggle submission

In [9]:
submission = pd.DataFrame({'PassengerId': passenger_id, 'Survived': test_pred})
out_path = os.path.join(SUB, 'submission_linear_regression.csv')
submission.to_csv(out_path, index=False)

print('Saved:', out_path)
print('Shape:', submission.shape)
print(submission['Survived'].value_counts())
submission.head()

Saved: ../submissions/submission_linear_regression.csv
Shape: (418, 2)
Survived
0    262
1    156
Name: count, dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1


## 13. Conclusion (demo)

- Used **only `LinearRegression`** as required — no classifier.
- Key features: `Sex`, `Pclass`, `Age`, `Fare`, `Embarked`, `FamilySize`, `IsAlone`, `Title`, `HasCabin`, `Deck`, `TicketGroupSize`.
- Missing `Age` filled with grouped medians (Title + Pclass), not a single global mean.
- Since the regressor outputs continuous values, the decision threshold was tuned with repeated stratified cross-validation (~0.83 CV accuracy).
- Retrained on the full training set and produced `submissions/submission_linear_regression.csv` (418 rows).